# 02 — Model Training: Faster R-CNN with ResNet Backbones

**Covers:**
- `notes/02-advanced-deep-learning/ch01-residual-networks/` — ResNet architecture as the detection backbone
- `notes/02-advanced-deep-learning/ch03-object-detection/` or `ch04` — Faster R-CNN architecture (RPN + ROI head)
- `notes/02-advanced-deep-learning/ch05-mixed-precision/` — `autocast` + `GradScaler` training loop

**Goal:** Train a Faster R-CNN detector, log all loss components per epoch, plot convergence curves, and compare three backbone choices on accuracy vs. parameter count.

**Prerequisites:**
- Completed `01_data_exploration.ipynb` — understand the dataset structure
- CUDA GPU recommended (or set `QUICK_MODE=True` for a CPU-runnable dry run)
- `torchvision >= 0.13`, `pycocotools`

**`QUICK_MODE`:** Limits training to 2 epochs and 50 images so the notebook completes in ~5 min on CPU.

In [ ]:
# ── Imports & training configuration ─────────────────────────────────────────
import sys, os, time, json, random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from pathlib import Path
from torch.cuda.amp import GradScaler, autocast
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

# QUICK_MODE=True: 2 epochs, 50 synthetic images, no real dataset needed
# QUICK_MODE=False: full training — requires real COCO-format data & GPU
QUICK_MODE = False

# Central config dict — mirrors what src/models.py TrainingConfig expects
CONFIG = {
    'num_epochs':   2 if QUICK_MODE else 20,
    'batch_size':   2 if QUICK_MODE else 4,
    'learning_rate': 5e-4,
    'weight_decay': 1e-4,
    'num_classes':  6,       # 5 retail categories + background
    'backbone':     'resnet50_fpn',  # options: resnet50_fpn | resnet101_fpn | efficientnet_b3
    'device':       'cuda' if torch.cuda.is_available() else 'cpu',
    'data_dir':     Path('../data/coco'),
    'checkpoint_dir': Path('../models'),
    'warmup_epochs': 1,      # LR linearly ramps for this many epochs
}

torch.manual_seed(42)
np.random.seed(42)
CONFIG['checkpoint_dir'].mkdir(parents=True, exist_ok=True)
print(f"Device: {CONFIG['device']} | QUICK_MODE: {QUICK_MODE}")
print(f"Config: {CONFIG['num_epochs']} epochs, batch_size={CONFIG['batch_size']}")

## 1. Backbone Choice — ResNet-50 vs ResNet-101 vs EfficientNet

**The tradeoff:** ResNet-50 is the standard workhorse — well-understood, fast to train, good mAP. ResNet-101 adds ~20M parameters for ~1-2% mAP gain. EfficientNet-B3/B4 achieves similar accuracy with fewer FLOPs but requires custom FPN integration in older torchvision.

**Transfer learning strategy (see notes ch01):**
- We load ImageNet-pretrained weights for the backbone (free feature extractor)
- We **freeze** backbone layers 1–3 for the first 5 epochs (only train detection head)
- Then **unfreeze** the full network for fine-tuning (backbone learns task-specific features)

This two-stage approach converges faster and prevents catastrophic forgetting of ImageNet features.

In [ ]:
# ── Load pretrained Faster R-CNN and adapt the classifier head ───────────────
# torchvision provides a Faster R-CNN with ResNet-50 + FPN backbone
# pre-trained on COCO 2017 (80 classes). We replace the box predictor
# head to match our num_classes.

def build_fasterrcnn(num_classes, backbone_name='resnet50_fpn'):
    """Build Faster R-CNN with a torchvision backbone.

    For resnet101_fpn or efficientnet variants, you would substitute
    the corresponding builder function. The interface stays identical.
    """
    if backbone_name == 'resnet50_fpn':
        # Weights=DEFAULT loads the latest COCO pretrained checkpoint
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
    elif backbone_name == 'resnet101_fpn':
        # NOTE: requires torchvision >= 0.15 — falls back to resnet50 here
        try:
            from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2
            from torchvision.models.detection import FasterRCNN_ResNet50_FPN_V2_Weights
            model = fasterrcnn_resnet50_fpn_v2(weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
            print('Using ResNet50-FPN-v2 as resnet101_fpn proxy')
        except ImportError:
            model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
            print('Fell back to resnet50_fpn')
    else:
        model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)

    # Replace the box classifier — this is task-specific (num_classes includes background)
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    return model

model = build_fasterrcnn(CONFIG['num_classes'], CONFIG['backbone'])

# Freeze backbone layers 1-3 for warmup training (only train RPN + ROI head)
layers_to_freeze = ['backbone.body.layer1', 'backbone.body.layer2', 'backbone.body.layer3']
frozen_count = 0
for name, param in model.named_parameters():
    if any(name.startswith(layer) for layer in layers_to_freeze):
        param.requires_grad = False
        frozen_count += 1

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Backbone: {CONFIG["backbone"]}')
print(f'Total params: {total/1e6:.1f}M | Trainable (head only): {trainable/1e6:.1f}M')
print(f'Frozen params: {frozen_count} tensors (backbone layers 1-3)')

model = model.to(CONFIG['device'])

## 2. Training Loop — What Are We Optimizing?

Faster R-CNN has **four loss components** that are summed automatically by `torchvision`:

| Loss | What it measures |
|------|------------------|
| `loss_classifier` | Cross-entropy on ROI class predictions |
| `loss_box_reg` | Smooth-L1 on ROI bbox offsets (positive anchors only) |
| `loss_objectness` | Binary cross-entropy in the RPN (anchor is object vs background) |
| `loss_rpn_box_reg` | Smooth-L1 on RPN anchor refinement |

**Mixed precision (`autocast`):** Runs forward pass in FP16, backward in FP32. The `GradScaler` multiplies the loss by a large constant before `backward()` to prevent FP16 underflow, then un-scales gradients before the optimizer step. Net effect: ~2x faster on Volta/Ampere GPUs, no accuracy loss.

**LR warmup:** We ramp learning rate from 0 → `learning_rate` over `warmup_epochs` to stabilise the randomly-initialised detection head before training the frozen backbone layers.

In [ ]:
# ── Training loop with loss logging ──────────────────────────────────────────
# NOTE: This cell requires GPU for full training. Set QUICK_MODE=True for a
# CPU dry-run (2 epochs, synthetic data, verifies the loop runs correctly).

from torch.utils.data import DataLoader, Dataset

class SyntheticCOCODataset(Dataset):
    """Synthetic dataset that produces COCO-format targets for loop testing.
    Replace with COCODataLoader from src/data.py when real data is available.
    """
    def __init__(self, n=50, num_classes=6, img_size=416):
        self.n = n
        self.num_classes = num_classes
        self.img_size = img_size

    def __len__(self): return self.n

    def __getitem__(self, idx):
        torch.manual_seed(idx)
        img = torch.rand(3, self.img_size, self.img_size)
        n_obj = torch.randint(1, 5, (1,)).item()
        # torchvision detection models expect [x1, y1, x2, y2]
        boxes = torch.rand(n_obj, 4)
        boxes[:, 2:] = boxes[:, :2] + torch.rand(n_obj, 2) * 0.3 + 0.05
        boxes = boxes.clamp(0, 1) * self.img_size
        boxes[:, 2] = boxes[:, 2].clamp(min=boxes[:, 0] + 1)
        boxes[:, 3] = boxes[:, 3].clamp(min=boxes[:, 1] + 1)
        labels = torch.randint(1, self.num_classes, (n_obj,))
        return img, {'boxes': boxes.float(), 'labels': labels}

def collate_fn(batch):
    images, targets = zip(*batch)
    return list(images), list(targets)

n_images = 50 if QUICK_MODE else 1000  # replace with real dataset size
dataset = SyntheticCOCODataset(n=n_images, num_classes=CONFIG['num_classes'])
loader  = DataLoader(dataset, batch_size=CONFIG['batch_size'],
                     shuffle=True, collate_fn=collate_fn, num_workers=0)

optimizer = torch.optim.SGD(
    [p for p in model.parameters() if p.requires_grad],
    lr=CONFIG['learning_rate'],
    momentum=0.9,
    weight_decay=CONFIG['weight_decay'],
)
scaler = GradScaler(enabled=(CONFIG['device'] == 'cuda'))

# Warmup scheduler: linearly ramp LR over warmup_epochs
warmup_iters = CONFIG['warmup_epochs'] * len(loader)
def warmup_lambda(step):
    return min(1.0, step / warmup_iters) if step < warmup_iters else 1.0
lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_lambda)

# Training loop — logs all four loss components per epoch
history = {'epoch': [], 'loss_total': [], 'loss_classifier': [],
           'loss_box_reg': [], 'loss_objectness': [], 'loss_rpn_box_reg': []}

model.train()
for epoch in range(1, CONFIG['num_epochs'] + 1):
    # Unfreeze backbone after warmup epochs
    if epoch == CONFIG['warmup_epochs'] + 1:
        for param in model.parameters():
            param.requires_grad = True
        print(f'Epoch {epoch}: Unfreezing backbone — full fine-tuning begins')

    epoch_losses = {k: 0.0 for k in ['loss_classifier', 'loss_box_reg',
                                       'loss_objectness', 'loss_rpn_box_reg']}
    for images, targets in loader:
        images  = [img.to(CONFIG['device']) for img in images]
        targets = [{k: v.to(CONFIG['device']) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        with autocast(enabled=(CONFIG['device'] == 'cuda')):
            loss_dict = model(images, targets)  # returns dict of 4 losses
            total_loss = sum(loss_dict.values())

        scaler.scale(total_loss).backward()
        scaler.step(optimizer)
        scaler.update()
        lr_scheduler.step()

        for k in epoch_losses:
            epoch_losses[k] += loss_dict.get(k, torch.tensor(0.0)).item()

    avg = {k: v / len(loader) for k, v in epoch_losses.items()}
    avg_total = sum(avg.values())
    history['epoch'].append(epoch)
    history['loss_total'].append(avg_total)
    for k in epoch_losses:
        history[k].append(avg[k])
    print(f'Epoch [{epoch:02d}/{CONFIG["num_epochs"]}]  '
          f'total={avg_total:.4f}  cls={avg["loss_classifier"]:.3f}  '
          f'box={avg["loss_box_reg"]:.3f}  obj={avg["loss_objectness"]:.3f}  '
          f'rpn={avg["loss_rpn_box_reg"]:.3f}')

# Save checkpoint for later notebooks
ckpt_path = CONFIG['checkpoint_dir'] / 'fasterrcnn_trained.pth'
torch.save({'model_state_dict': model.state_dict(), 'config': CONFIG,
            'history': history}, ckpt_path)
print(f'\nCheckpoint saved to {ckpt_path}')

## 3. Training Curves — Diagnosing Convergence

**What to look for:**
- `loss_objectness` should drop quickly (easy: is there *any* object here?)
- `loss_classifier` should drop after objectness stabilises
- `loss_box_reg` drops last — precise localisation is the hardest sub-task
- **Vertical line at LR warmup end** marks where the backbone unfreezes — expect a temporary loss spike

**Diagnosing problems:**
- Loss plateaus immediately → LR too low, or batch norm stats frozen
- Loss diverges → LR too high; reduce by 10x
- `loss_objectness` stuck at 0.7+ → anchors don't match object scales (revisit notebook 01 bbox analysis)

In [ ]:
# ── Plot training curves ──────────────────────────────────────────────────────
epochs = history['epoch']
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: total loss + component breakdown
axes[0].plot(epochs, history['loss_total'], 'k-', linewidth=2, label='Total')
for key, color in [('loss_classifier', 'steelblue'), ('loss_box_reg', 'coral'),
                    ('loss_objectness', 'green'),    ('loss_rpn_box_reg', 'purple')]:
    axes[0].plot(epochs, history[key], linestyle='--', color=color,
                 label=key.replace('loss_', ''), alpha=0.8)

# Mark LR warmup end
warmup_end = CONFIG['warmup_epochs']
if warmup_end < CONFIG['num_epochs']:
    axes[0].axvline(warmup_end + 0.5, color='orange', linestyle=':', linewidth=1.5,
                    label=f'Backbone unfrozen (epoch {warmup_end+1})')

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss Components')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

# Right: loss component shares over time
component_keys = ['loss_classifier', 'loss_box_reg', 'loss_objectness', 'loss_rpn_box_reg']
totals = np.array(history['loss_total'])
shares = {k: np.array(history[k]) / (totals + 1e-9) for k in component_keys}
bottom = np.zeros(len(epochs))
colors = ['steelblue', 'coral', 'green', 'purple']
for key, color in zip(component_keys, colors):
    axes[1].bar(epochs, shares[key], bottom=bottom, label=key.replace('loss_', ''),
                color=color, alpha=0.8)
    bottom += shares[key]
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Share of Total Loss')
axes[1].set_title('Loss Component Shares Over Training')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4. Evaluation — mAP@0.5 and mAP@0.5:0.95

**`mAP@0.5`** is the standard PASCAL VOC metric — a detection counts as correct if its IoU with a ground truth box exceeds 0.5.

**`mAP@0.5:0.95`** is the COCO metric — average over IoU thresholds [0.5, 0.55, ..., 0.95]. Rewards tighter localisation; typically 20-30% lower than mAP@0.5 for the same model.

**ProductionCV constraints (see notes ch01 intro):**
- Target: mAP@0.5 ≥ 85%
- The ResNet-50 baseline typically reaches 38-40% mAP@0.5:0.95 (≈ 60% mAP@0.5) on COCO 80-class. On our 5-class retail dataset with fewer, cleaner classes, we expect significantly higher values.

In [ ]:
# ── COCO evaluation using pycocotools ────────────────────────────────────────
# NOTE: This cell requires pycocotools and a real validation set with ground
# truth annotations. With QUICK_MODE=True / synthetic data it computes
# approximate metrics using torchvision's mean_average_precision.
# For production evaluation: replace val_loader with real COCO val split.

try:
    from torchvision.ops import box_iou
    HAS_TV = True
except ImportError:
    HAS_TV = False

def simple_map_at_threshold(preds, targets, iou_threshold=0.5):
    """Compute approximate mAP at a single IoU threshold.

    This is a simplified implementation for quick feedback.
    For publication-quality metrics use pycocotools.COCO + COCOeval.
    """
    if not HAS_TV:
        return float('nan')
    tp_total, fp_total, gt_total = 0, 0, 0
    for pred, target in zip(preds, targets):
        p_boxes = pred.get('boxes', torch.zeros(0, 4))
        t_boxes = target.get('boxes', torch.zeros(0, 4))
        gt_total += len(t_boxes)
        if len(p_boxes) == 0 or len(t_boxes) == 0:
            fp_total += len(p_boxes)
            continue
        iou_matrix = box_iou(p_boxes, t_boxes)
        matched = set()
        for pred_idx in range(len(p_boxes)):
            best_iou, best_gt = iou_matrix[pred_idx].max(0)
            if best_iou.item() >= iou_threshold and best_gt.item() not in matched:
                tp_total += 1
                matched.add(best_gt.item())
            else:
                fp_total += 1
    precision = tp_total / (tp_total + fp_total + 1e-9)
    recall    = tp_total / (gt_total + 1e-9)
    f1        = 2 * precision * recall / (precision + recall + 1e-9)
    return {'precision': precision, 'recall': recall, 'f1': f1,
            'tp': tp_total, 'fp': fp_total, 'gt': gt_total}

# Run inference on the validation set (synthetic here)
val_dataset = SyntheticCOCODataset(n=20 if QUICK_MODE else 200,
                                   num_classes=CONFIG['num_classes'])
val_loader  = DataLoader(val_dataset, batch_size=1,
                          shuffle=False, collate_fn=collate_fn)
model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for images, targets in val_loader:
        images = [img.to(CONFIG['device']) for img in images]
        preds  = model(images)  # in eval mode returns list of dicts
        all_preds.extend([{k: v.cpu() for k, v in p.items()} for p in preds])
        all_targets.extend(targets)

metrics_50   = simple_map_at_threshold(all_preds, all_targets, iou_threshold=0.50)
metrics_75   = simple_map_at_threshold(all_preds, all_targets, iou_threshold=0.75)

print(f'IoU@0.50 → Precision: {metrics_50["precision"]:.3f}, '
      f'Recall: {metrics_50["recall"]:.3f}, F1: {metrics_50["f1"]:.3f}')
print(f'IoU@0.75 → Precision: {metrics_75["precision"]:.3f}, '
      f'Recall: {metrics_75["recall"]:.3f}, F1: {metrics_75["f1"]:.3f}')
print('\n[NOTE] For official mAP@0.5:0.95 use pycocotools.COCOeval on real annotations.')

## 5. Architecture Comparison — What Gains What?

The table below summarises the accuracy-efficiency frontier for three backbone choices. Numbers are from the torchvision model zoo (COCO 2017 val) — your retail dataset results will differ but the *relative ordering* holds.

**Key insight:** ResNet-101 gives a modest mAP boost (+1-2%) at the cost of 30% more parameters and 40% more inference time. For the ProductionCV constraint of <50ms/frame on a Jetson Nano, ResNet-50 is often the better choice. EfficientNet-B3 offers the best FLOPs/accuracy tradeoff but requires more careful tuning.

In [ ]:
# ── Architecture comparison table ────────────────────────────────────────────
# Benchmark inference on current device for the loaded model;
# fill remaining rows from the torchvision model zoo benchmarks.
import pandas as pd

def measure_inference_ms(model, img_size=416, n_runs=20, device='cpu'):
    """Warmup then time inference over n_runs batches of 1 image."""
    dummy = [torch.rand(3, img_size, img_size).to(device)]
    model.eval()
    # Warmup passes remove CUDA kernel launch overhead
    with torch.no_grad():
        for _ in range(5):
            _ = model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.no_grad():
        for _ in range(n_runs):
            _ = model(dummy)
    if device == 'cuda':
        torch.cuda.synchronize()
    return (time.perf_counter() - start) / n_runs * 1000  # ms

our_params   = sum(p.numel() for p in model.parameters()) / 1e6
our_latency  = measure_inference_ms(model, device=CONFIG['device'])

# Table: backbone → params (M) → mAP@0.5:0.95 (COCO model zoo) → infer_ms
comparison = pd.DataFrame([
    {'backbone': 'ResNet-50-FPN (this run)',  'params_M': round(our_params, 1),
     'mAP_coco': '37.0 (zoo)',  'infer_ms': round(our_latency, 1),
     'notes': 'Good balance; default choice'},
    {'backbone': 'ResNet-50-FPN-v2',          'params_M': 43.7,
     'mAP_coco': '46.7 (zoo)',  'infer_ms': '~65ms (GPU)',
     'notes': 'v2 training recipe; +9% mAP vs v1'},
    {'backbone': 'ResNet-101-FPN',             'params_M': 60.2,
     'mAP_coco': '42.0 (zoo)',  'infer_ms': '~85ms (GPU)',
     'notes': 'Larger; marginal gain vs cost'},
    {'backbone': 'MobileNetV3-Large-FPN',      'params_M': 19.4,
     'mAP_coco': '32.8 (zoo)',  'infer_ms': '~28ms (GPU)',
     'notes': 'Edge deployment candidate (ch04)'},
])
print(comparison.to_string(index=False))
print(f'\nMeasured latency on {CONFIG["device"]}: {our_latency:.1f} ms/image')